# CS 553 - Neural Networks Project 1

This notebooks contains code that aims to mimic the results of the MedMNIST V2 paper. We will reproduce and train a ResNet-18 model on the VessleMNIST dataset utilizing 3D convolutions. The module we aim to use is pytorch.

## Image Preprocessing

In [3]:
pip install medmnist

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [medmnist]
Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install acsconv

  Preparing metadata (setup.py) ... done
  DEPRECATION: Building 'acsconv' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'acsconv'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  Created wheel for acsconv: filename=acsconv-0.1.1-py3-none-any.whl size=24239 sha256=cba53ac9bf1a9c85e00e14ba2eae48cc06f791a230886c3bfd8870625f8b5f19
  Stored in directory: /home/jovyan/.cache/pip/wheels/19/ef/95/02b235a700cd36e7bcf928352d2cba18f8ee46e590e1391e7f
Successfully built acsconv
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [acsconv]
Note: you may need to restart the kernel to use updated packages.


In [69]:
#import packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch 
from medmnist import VesselMNIST3D
from torch.utils.data import Dataset
from acsconv.converters import ACSConverter, Conv3dConverter
import torchvision.models as models 
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import seaborn as sns
import random

In [70]:
def set_seed(seed=42):
    """Set seeds for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if using multi-GPU
    
    # Make CuDNN deterministic (may reduce performance)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Call the function at the start
set_seed(42)

In [71]:
#Import VesselMNIST training dataset 
train_dataset = VesselMNIST3D(split="train", download=True)
train_dataset

Dataset VesselMNIST3D of size 28 (vesselmnist3d)
    Number of datapoints: 1335
    Root location: /home/jovyan/.medmnist
    Split: train
    Task: binary-class
    Number of channels: 1
    Meaning of labels: {'0': 'vessel', '1': 'aneurysm'}
    Number of samples: {'train': 1335, 'val': 191, 'test': 382}
    Description: The VesselMNIST3D is based on an open-access 3D intracranial aneurysm dataset, IntrA, containing 103 3D models (meshes) of entire brain vessels collected by reconstructing MRA images. 1,694 healthy vessel segments and 215 aneurysm segments are generated automatically from the complete models. We fix the non-watertight mesh with PyMeshFix and voxelize the watertight mesh with trimesh into 28×28×28 voxels. We split the source dataset with a ratio of 7:1:2 into training, validation and test set.
    License: CC BY 4.0

In [72]:
#import VessleMNIST val dataset
val_dataset = VesselMNIST3D(split="val", download= True)
val_dataset

Dataset VesselMNIST3D of size 28 (vesselmnist3d)
    Number of datapoints: 191
    Root location: /home/jovyan/.medmnist
    Split: val
    Task: binary-class
    Number of channels: 1
    Meaning of labels: {'0': 'vessel', '1': 'aneurysm'}
    Number of samples: {'train': 1335, 'val': 191, 'test': 382}
    Description: The VesselMNIST3D is based on an open-access 3D intracranial aneurysm dataset, IntrA, containing 103 3D models (meshes) of entire brain vessels collected by reconstructing MRA images. 1,694 healthy vessel segments and 215 aneurysm segments are generated automatically from the complete models. We fix the non-watertight mesh with PyMeshFix and voxelize the watertight mesh with trimesh into 28×28×28 voxels. We split the source dataset with a ratio of 7:1:2 into training, validation and test set.
    License: CC BY 4.0

In [73]:
#import VessleMNIST testing dataset
test_dataset = VesselMNIST3D(split="test", download= True)
test_dataset

Dataset VesselMNIST3D of size 28 (vesselmnist3d)
    Number of datapoints: 382
    Root location: /home/jovyan/.medmnist
    Split: test
    Task: binary-class
    Number of channels: 1
    Meaning of labels: {'0': 'vessel', '1': 'aneurysm'}
    Number of samples: {'train': 1335, 'val': 191, 'test': 382}
    Description: The VesselMNIST3D is based on an open-access 3D intracranial aneurysm dataset, IntrA, containing 103 3D models (meshes) of entire brain vessels collected by reconstructing MRA images. 1,694 healthy vessel segments and 215 aneurysm segments are generated automatically from the complete models. We fix the non-watertight mesh with PyMeshFix and voxelize the watertight mesh with trimesh into 28×28×28 voxels. We split the source dataset with a ratio of 7:1:2 into training, validation and test set.
    License: CC BY 4.0

In [74]:
#Look at the first few samples of the dataset
for i in range(5):
    sample_image, sample_label = train_dataset[i]
    print(f'sample image', sample_image, 'sample_label', sample_label)

sample image [[[[0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   ...
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]]

  [[0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   ...
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]]

  [[0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   ...
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]]

  ...

  [[0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   ...
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]]

  [[0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   ...
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]]

  [[0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   ...
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 

In [75]:
sample_image, sample_label = train_dataset[0]
sample_image

array([[[[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],

        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],

        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],

        ...,

        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
    

In [76]:
#Create class that expands the image channel from 1 -> 3
#Data augmentation and normalization
class ExpandedVesselMNIST(Dataset):
    def __init__(self, base_dataset, is_training = True):
        self.base_dataset = base_dataset
        self.is_training = is_training
    
    def __len__(self):
        return len(self.base_dataset)
    
    def __getitem__(self, idx):
        image, label = self.base_dataset[idx]
        tensor_image = torch.from_numpy(image).float()

        if self.is_training:
            rand_factor = torch.rand(1).item()
            tensor_image = tensor_image * rand_factor
        else:
            tensor_image = tensor_image * 0.5

        expanded_image = tensor_image.repeat(3, 1, 1, 1)

        if isinstance(label, np.ndarray):
            label = label.squeeze()
            
        return expanded_image, torch.tensor(label).long()

In [77]:
#Expand image channels for training dataset 
mod_train_set = ExpandedVesselMNIST(train_dataset, is_training=True)
mod_val_set = ExpandedVesselMNIST(val_dataset, is_training=False)
mod_test_set = ExpandedVesselMNIST(test_dataset, is_training=False)

batch_size = 32
train_loader = DataLoader(
    mod_train_set,
    batch_size=32,
    shuffle = True,
    num_workers=2,
    pin_memory=True)

val_loader = DataLoader(
    mod_val_set,
    batch_size= batch_size,
    shuffle = False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    mod_test_set,
    batch_size= batch_size,
    shuffle = False,
    num_workers=2,
    pin_memory=True
)



# Model Architecture and Training: ResNet-18 with 3D Convolutions

In [78]:
#Import resnet18 utilizing an untrained neural network 
resnet18 = models.resnet18(pretrained=False)
#Convert model into 3D convolutional neural network
model_3d = Conv3dConverter(resnet18)

In [79]:
#Change output nodes to 2 for binary classification task
num_classes = 2
model_3d.fc = torch.nn.Linear(model_3d.fc.in_features, num_classes)

In [80]:
#Cross Entropy Loss for classification tasks, adam optimizer to match with paper specification
criterion = torch.nn.CrossEntropyLoss()
#Pass through model parameters to optimizers, pass starting learning rate of 0.001
optimizer = torch.optim.Adam(model_3d.parameters(), lr=0.001)
#Implement a learning rate scheduler to change lr at epoch 50 and 75
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[50, 75], gamma = 0.1)

In [81]:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_3d = model_3d.to(device)
print(f"Using device: {device}")

Using device: cuda


In [82]:
#Run for 100 epochs
num_epochs = 100

#early stopping parameters
patience = 10
best_val_loss = float('inf')
epochs_no_improve = 0
early_stop = False

#For each of the epochs
for epoch in range(num_epochs):

    if early_stop:
        print(f"Early stopping is triggered at epoch {epoch}")
        break
    
    #Set the model to training mode
    model_3d.train()
    train_loss = 0.0
    #Load each batch per epoch, batch size is 32
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        #zero out the gradients per session
        optimizer.zero_grad()
        #Plug in inputs to the NN and get output
        outputs = model_3d(inputs)
        #Feed actual and predicted values into loss function
        loss = criterion(outputs, labels)
        #Backpropogate to compute derivatives
        loss.backward()
        #Updated model weights
        optimizer.step()
        train_loss += loss.item()
    avg_train_loss = train_loss / len(train_loader)

    model_3d.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            #Forward pass on the data
            outputs = model_3d(inputs)

            #Get the loss
            loss = criterion(outputs, labels)

            #Add up validation loss in epoch
            val_loss += loss.item()

            #get preidctions
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)

            correct += (predicted==labels).sum().item()
    avg_val_loss = val_loss / len(val_loader)
    accuracy = correct / total * 100
    # Print both training and validation metrics
    print(f"Epoch {epoch+1:3d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {accuracy:.2f}%")
    

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_no_improve = 0
        torch.save(model_3d.state_dict(), 'best_model.pth')
    else:
        epochs_no_improve+=1

        if epochs_no_improve >= patience:
            early_stop = True
    scheduler.step()

Epoch   1 | Train Loss: 0.4853 | Val Loss: 0.3449 | Val Acc: 88.48%
Epoch   2 | Train Loss: 0.3573 | Val Loss: 0.3459 | Val Acc: 88.48%
Epoch   3 | Train Loss: 0.3382 | Val Loss: 0.3281 | Val Acc: 88.48%
Epoch   4 | Train Loss: 0.3627 | Val Loss: 0.3040 | Val Acc: 88.48%
Epoch   5 | Train Loss: 0.3213 | Val Loss: 0.2799 | Val Acc: 89.53%
Epoch   6 | Train Loss: 0.2816 | Val Loss: 0.3162 | Val Acc: 89.01%
Epoch   7 | Train Loss: 0.2547 | Val Loss: 0.2837 | Val Acc: 90.05%
Epoch   8 | Train Loss: 0.1791 | Val Loss: 0.5404 | Val Acc: 88.48%
Epoch   9 | Train Loss: 0.2077 | Val Loss: 0.3704 | Val Acc: 88.48%
Epoch  10 | Train Loss: 0.1550 | Val Loss: 0.3252 | Val Acc: 88.48%
Epoch  11 | Train Loss: 0.1407 | Val Loss: 0.3205 | Val Acc: 90.58%
Epoch  12 | Train Loss: 0.1255 | Val Loss: 0.5191 | Val Acc: 84.29%
Epoch  13 | Train Loss: 0.1205 | Val Loss: 0.5653 | Val Acc: 89.53%
Epoch  14 | Train Loss: 0.1060 | Val Loss: 0.4014 | Val Acc: 86.91%
Epoch  15 | Train Loss: 0.0749 | Val Loss: 0.349

In [84]:
model_3d.load_state_dict(torch.load('best_model.pth'))
model_3d.eval()

test_loss = 0.0
test_correct = 0
test_total = 0

all_predictions = []
all_labels = []
all_probabilites =[]
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model_3d(inputs)
        

        probabilites = torch.nn.functional.softmax(outputs,dim=1)
        all_probabilites.extend(probabilites[:, 1].cpu().numpy())

        loss = criterion(outputs, labels)
        test_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        test_total += labels.size(0)
        test_correct += (labels == predicted).sum().item()
        all_predictions.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

avg_test_loss = test_loss / len(test_loader)
test_accuracy = 100 * test_correct / test_total


auc_score = roc_auc_score(all_labels, all_probabilites)

print("=" * 50)
print("TEST SET RESULTS")
print("=" * 50)
print(f"Test Loss: {avg_test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.2f}%")
print(f"Test AUC: {auc_score:.4f}")
print(f"Correct Predictions: {test_correct}/{test_total}")

# Classification report

TEST SET RESULTS
Test Loss: 0.2726
Test Accuracy: 88.74%
Test AUC: 0.8521
Correct Predictions: 339/382
